# DSA 210 — Impact of Academic Calendar & Physical Activity on Personal Biochemistry
**Author:** Defne Akman  
**Dataset:** e-nabız blood tests (2022–2025) · Apple Health step counts · Sabancı University Academic Calendar  
**Deadline:** April 14, 2026 — Data Collection, EDA & Hypothesis Testing


## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Load pre-processed data
blood_df = pd.read_csv('../data/processed/blood_with_labels.csv', parse_dates=['date'])
steps_df  = pd.read_csv('../data/processed/daily_steps.csv', parse_dates=['date'])
steps_df  = steps_df[steps_df['date'] >= '2022-01-01']

print(f"Blood test records: {len(blood_df)}")
print(f"Daily step records: {len(steps_df)}")
print(f"Blood test date range: {blood_df['date'].min().date()} → {blood_df['date'].max().date()}")
print(f"Step data date range:  {steps_df['date'].min().date()} → {steps_df['date'].max().date()}")
blood_df[['date','period_name','LDL','HDL','Total_Chol','Triglycerides','Fasting_Glucose','avg_steps_14d']]

## 2. Blood Data — Extracted from e-nabız

Key metabolic parameters extracted from 20 blood tests (2022–2025):

In [ ]:
# Blood test summary statistics
print("=== Blood Marker Summary Statistics ===")
markers = ['LDL','HDL','Total_Chol','Triglycerides','Fasting_Glucose']
print(blood_df[markers].describe().round(1))

# By academic period
print("\n=== Mean Values by Academic Period ===")
print(blood_df.groupby('period_name')[markers].mean().round(1))

## 3. Step Count Data — Apple Health

Extracted from Apple Health XML export (78,391 raw records → aggregated to daily totals):

In [ ]:
print("=== Daily Step Count Summary ===")
print(steps_df['daily_steps'].describe().round(0))

# Weekly rolling average plot
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(steps_df['date'], steps_df['daily_steps'], color='#E0E0E0', lw=0.4, alpha=0.5)
ax.plot(steps_df['date'], steps_df['daily_steps'].rolling(7).mean(), color='#4285F4', lw=2)
ax.set_title('Daily Steps with 7-day Rolling Average (2022–2026)')
ax.set_ylabel('Steps')
plt.tight_layout()
plt.show()

## 4. Academic Calendar Labeling

Periods labeled based on Sabancı University official calendar:
- **Regular** (0): active semester, lecture weeks
- **Finals** (1): exam weeks (~2 weeks per semester)
- **Holiday** (2): inter-semester breaks, summer

In [ ]:
print("Blood tests by academic period:")
print(blood_df['period_name'].value_counts())
print("\nNote: No blood tests fall exactly during finals windows —")
print("comparison will be Regular (high-stress proxy) vs Holiday (low-stress proxy)")

## 5. EDA — Visualizations

See `fig1_overview.png`, `fig2_eda.png`, `fig3_hypothesis.png` for full visualizations.

In [ ]:
from IPython.display import Image
Image('fig1_overview.png')

## 6. Hypothesis Testing

### H1: Do blood markers differ between regular semester and holiday periods?
**H₀:** No difference in blood marker means between periods  
**H₁:** Blood markers differ between regular and holiday periods  
**Test:** Mann-Whitney U (non-parametric, small n)

In [ ]:
regular = blood_df[blood_df['period_name'] == 'regular']
holiday = blood_df[blood_df['period_name'] == 'holiday']

print(f"Regular period samples: {len(regular)}")
print(f"Holiday period samples:  {len(holiday)}")
print()

results = []
for col in ['LDL','HDL','Triglycerides','Total_Chol','Fasting_Glucose']:
    r_vals = regular[col].dropna().values
    h_vals = holiday[col].dropna().values
    if len(r_vals) < 2 or len(h_vals) < 2:
        continue
    stat, p = stats.mannwhitneyu(r_vals, h_vals, alternative='two-sided')
    results.append({'Marker': col, 'Regular μ': round(np.mean(r_vals),1),
                    'Holiday μ': round(np.mean(h_vals),1),
                    'Δ (Holiday-Regular)': round(np.mean(h_vals)-np.mean(r_vals),1),
                    'p-value': round(p,4), 'Significant (p<0.05)': p<0.05})

pd.DataFrame(results).set_index('Marker')

### H2: Do step counts correlate with blood markers?
**H₀:** No linear correlation between avg steps and blood markers  
**H₁:** Significant correlation exists  
**Test:** Pearson correlation (14-day avg steps before each test)

In [ ]:
corr_results = []
for col in ['LDL','HDL','Triglycerides','Total_Chol','Fasting_Glucose']:
    sub = blood_df[['avg_steps_14d', col]].dropna()
    if len(sub) < 3:
        continue
    r, p = stats.pearsonr(sub['avg_steps_14d'], sub[col])
    corr_results.append({'Marker': col, 'r': round(r,3), 'p-value': round(p,4),
                         'Direction': '↑ with steps' if r>0 else '↓ with steps',
                         'Significant (p<0.05)': p<0.05})
pd.DataFrame(corr_results).set_index('Marker')

### H3: Are step counts significantly lower during finals vs holidays?
**H₀:** No difference in step counts between finals and holiday periods  
**H₁:** Step counts are lower during finals (higher stress → less movement)  
**Test:** Mann-Whitney U

In [ ]:
finals_windows = [
    ('2023-01-07','2023-01-20'), ('2023-05-27','2023-06-09'),
    ('2024-01-06','2024-01-19'), ('2024-05-25','2024-06-07'),
    ('2025-01-04','2025-01-17'), ('2025-05-24','2025-06-06'),
    ('2026-01-10','2026-01-23'),
]
holiday_windows = [
    ('2023-01-21','2023-02-20'), ('2023-06-10','2023-09-25'),
    ('2024-01-20','2024-02-12'), ('2024-06-08','2024-09-23'),
    ('2025-01-18','2025-02-10'), ('2025-06-07','2025-09-22'),
]

def get_steps(windows):
    vals = []
    for s, e in windows:
        mask = (steps_df['date'] >= pd.Timestamp(s)) & (steps_df['date'] <= pd.Timestamp(e))
        vals.extend(steps_df[mask]['daily_steps'].tolist())
    return np.array(vals)

finals_steps  = get_steps(finals_windows)
holiday_steps = get_steps(holiday_windows)
stat, p = stats.mannwhitneyu(finals_steps, holiday_steps, alternative='two-sided')

print(f"Finals period  — n={len(finals_steps):4d} days, mean = {np.mean(finals_steps):,.0f} steps/day")
print(f"Holiday period — n={len(holiday_steps):4d} days, mean = {np.mean(holiday_steps):,.0f} steps/day")
print(f"Difference: {np.mean(holiday_steps)-np.mean(finals_steps):+.0f} steps/day")
print(f"Mann-Whitney U statistic: {stat:.0f}")
print(f"p-value: {p:.4f}")
print(f"Conclusion: {'SIGNIFICANT difference (p < 0.05)' if p < 0.05 else f'Trend towards difference (p={p:.4f}), not significant at α=0.05'}") 

## 7. Conclusions & Limitations

### Key Findings
1. **H1 (Regular vs Holiday blood markers):** No statistically significant differences found (all p > 0.05). However, notable directional patterns exist — LDL and Total Cholesterol are *higher* in holiday periods, possibly due to reduced physical activity or dietary changes during breaks.

2. **H2 (Steps vs Blood markers):** Fasting Glucose shows the strongest negative correlation with step count (r=−0.436), suggesting more active periods associate with lower glucose — directionally consistent with exercise physiology. None reach significance given n=9.

3. **H3 (Steps — Finals vs Holiday):** Step counts are lower during finals (4,527 vs 5,916 steps/day, p=0.078). Near-significant trend: stress during finals may suppress physical activity.

### Limitations
- **Small blood test sample (n=9):** Blood tests were taken "when feeling off," introducing selection bias. Very low statistical power for detecting effects.
- **Temporal confounding:** Diet, sleep, and other lifestyle factors are unmeasured.
- **Period misalignment:** Blood tests rarely fall exactly on finals weeks, making direct finals-blood comparisons impossible with current data.

### Next Steps (for May 5 milestone)
- ML clustering of health states
- Time-series regression controlling for season/trend
- Collect additional contextual data (sleep, diet)
